# ⚡ Python Bit Operations — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**  
> Every integer in memory is just a row of **light switches** — on (`1`) or off (`0`).  
> Bit operations work directly on those switches — no addition, no loops.  
> This is why they're O(1) and why interviews love them.  
> The tricks look like magic until you see the pattern. Then they're obvious.

---

## 📋 Table of Contents

| # | Section |
|---|---|
| 1 | [The Binary Number System — Visual Model](#1) |
| 2 | [The Six Operators](#2) |
| 3 | [The Cancel Property — XOR's Superpower](#3) |
| 4 | [The Mask — Enforcing 32-bit Boundaries](#4) |
| 5 | [Pattern 1: Single Number (LC 136)](#5) |
| 6 | [Pattern 2: Missing Number (LC 268)](#6) |
| 7 | [Pattern 3: Number of 1 Bits (LC 191)](#7) |
| 8 | [Pattern 4: Counting Bits (LC 338)](#8) |
| 9 | [Pattern 5: Reverse Bits (LC 190)](#9) |
| 10 | [Pattern 6: Sum Without + or - (LC 371)](#10) |
| 11 | [The Bit Trick Decision Map](#11) |
| 12 | [Interview Cheat Sheet](#12) |

<a id='1'></a>
## 1. 🏗️ The Binary Number System — Visual Model

---

```
                    DECIMAL vs BINARY — 1 through 18

  Dec  Binary      Dec  Binary      Dec  Binary
  ───  ──────      ───  ──────      ───  ──────
   1   0001         7   0111        13   1101
   2   0010         8   1000        14   1110
   3   0011         9   1001        15   1111
   4   0100        10   1010        16   10000
   5   0101        11   1011        17   10001
   6   0110        12   1100        18   10010

  Notice:
  - Powers of 2 (1,2,4,8,16) always have exactly ONE '1' bit
  - Each bit position is worth double the one to its right
  - 4 bits covers 0-15. 5 bits covers 0-31. 32 bits covers 0-4 billion
```

---

### The bit position value map:

```
  Bit position:   7    6    5    4    3    2    1    0
  Value:        128   64   32   16    8    4    2    1

  Example — the number 13:
  Binary:         0    0    0    0    1    1    0    1
  Values:                              8    4         1
  Sum:  8 + 4 + 1 = 13  ✓

  Rule: bit at position i is worth 2^i
  Rule: n >> 1 drops the last bit (divides by 2, floor)
  Rule: n & 1 reads the last bit (0 or 1)
```

In [1]:
# See any number in binary
for n in [1, 2, 4, 8, 13, 16, 255]:
    print(f"{n:>3}  →  {bin(n)}  →  {n:08b}")

print()
# bin() gives '0b1101' — the 0b prefix means binary
# :08b format shows 8 bits zero-padded — cleaner for comparison

  1  →  0b1  →  00000001
  2  →  0b10  →  00000010
  4  →  0b100  →  00000100
  8  →  0b1000  →  00001000
 13  →  0b1101  →  00001101
 16  →  0b10000  →  00010000
255  →  0b11111111  →  11111111



<a id='2'></a>
## 2. 🔧 The Six Operators

---

```
OPERATOR   SYMBOL   WHAT IT DOES
─────────────────────────────────────────────────────────────
AND          &      Both bits must be 1 → output 1. Mask / isolate bits.
OR           |      Either bit is 1 → output 1. Set / combine bits.
XOR          ^      Bits are DIFFERENT → output 1. Toggle / cancel.
NOT          ~      Flip every bit. ~x = -(x+1) in Python.
LEFT SHIFT   <<     Multiply by 2^n. Moves bits left, fills 0s on right.
RIGHT SHIFT  >>     Divide by 2^n (floor). Moves bits right, drops last  .. Fills 0 to the left.
─────────────────────────────────────────────────────────────

Truth table for one bit pair (a, b):

  a  b  |  a&b  a|b  a^b
  ───────────────────────
  0  0  |   0    0    0
  0  1  |   0    1    1
  1  0  |   0    1    1
  1  1  |   1    1    0   ← XOR outputs 0 when SAME
```

In [2]:
a, b = 0b1010, 0b1100   # 10 and 12

print(f"a        = {a:04b}  ({a})")
print(f"b        = {b:04b}  ({b})")
print()
print(f"a & b    = {a&b:04b}  ({a&b})   AND  — only bits both have")
print(f"a | b    = {a|b:04b}  ({a|b})   OR   — bits either has")
print(f"a ^ b    = {a^b:04b}  ({a^b})   XOR  — bits only ONE has")
print(f"~a       = {~a}            NOT  — ~x = -(x+1)")
print(f"a << 1   = {a<<1:04b}  ({a<<1})  LEFT SHIFT  — multiply by 2")
print(f"a >> 1   = {a>>1:04b}  ({a>>1})   RIGHT SHIFT — divide by 2")

a        = 1010  (10)
b        = 1100  (12)

a & b    = 1000  (8)   AND  — only bits both have
a | b    = 1110  (14)   OR   — bits either has
a ^ b    = 0110  (6)   XOR  — bits only ONE has
~a       = -11            NOT  — ~x = -(x+1)
a << 1   = 10100  (20)  LEFT SHIFT  — multiply by 2
a >> 1   = 0101  (5)   RIGHT SHIFT — divide by 2


<a id='3'></a>
## 3. ⚡ The Cancel Property — XOR's Superpower

---

```
  THE TWO RULES THAT DRIVE EVERY XOR TRICK:

  1.  x ^ x = 0       same number cancels itself
  2.  x ^ 0 = x       XOR with zero does nothing

  XOR is also:
  - Commutative:   a ^ b = b ^ a
  - Associative:   (a ^ b) ^ c = a ^ (b ^ c)

  Translation: ORDER DOESN'T MATTER. Pairs always cancel
  regardless of where they appear in the array.

  Visual proof with [4, 1, 2, 1, 2]:

  Rearrange mentally: [1^1] ^ [2^2] ^ [4] = 0 ^ 0 ^ 4 = 4

  Linear proof (actual execution order):
  0 ^ 4 = 4
  4 ^ 1 = 5
  5 ^ 2 = 7
  7 ^ 1 = 6   ← 1 cancels
  6 ^ 2 = 4   ← 2 cancels
  result = 4  ✓

  INTERVIEW SIGNAL: "one element appears odd times, rest appear even"
  → XOR everything together. Pairs vanish. Odd one survives.
```

In [3]:
# Prove the cancel property
print("x ^ x = 0 ?")
for x in [1, 5, 13, 255, 1000]:
    print(f"  {x} ^ {x} = {x ^ x}")

print()
print("x ^ 0 = x ?")
for x in [1, 5, 13, 255, 1000]:
    print(f"  {x} ^ 0 = {x ^ 0}")

print()
# Order doesn't matter — pairs always cancel
nums = [4, 1, 2, 1, 2]
result = 0
for n in nums:
    result ^= n
    print(f"  XOR in {n}  →  result = {result}")
print(f"Final answer: {result}  (the lone survivor)")

x ^ x = 0 ?
  1 ^ 1 = 0
  5 ^ 5 = 0
  13 ^ 13 = 0
  255 ^ 255 = 0
  1000 ^ 1000 = 0

x ^ 0 = x ?
  1 ^ 0 = 1
  5 ^ 0 = 5
  13 ^ 0 = 13
  255 ^ 0 = 255
  1000 ^ 0 = 1000

  XOR in 4  →  result = 4
  XOR in 1  →  result = 5
  XOR in 2  →  result = 7
  XOR in 1  →  result = 6
  XOR in 2  →  result = 4
Final answer: 4  (the lone survivor)


<a id='4'></a>
## 4. 🔒 The Mask — Enforcing 32-bit Boundaries

---

```
  THE PROBLEM:
  Python integers have INFINITE precision. Hardware integers stop at 32 bits.
  LeetCode bit problems assume 32-bit signed integers.
  Negative numbers in Python create infinite carry chains → infinite loops.

  THE FIX: two constants that act as a 32-bit fence.

  MASK = 0xFFFFFFFF   →  1111 1111 ... 1111  (32 ones)
                         Chops off anything above bit 31.
                         x & MASK keeps only the bottom 32 bits.

  MAX  = 0x7FFFFFFF   →  0111 1111 ... 1111  (31 ones, sign bit is 0)
                         Highest positive 32-bit integer = 2,147,483,647
                         If result > MAX, bit 31 is set → it's NEGATIVE in 32-bit world

  CONVERTING BACK:
  ~(a ^ MASK)
  Step 1: a ^ MASK  → flip all 32 bits → strips the 32-bit encoding → small positive
  Step 2: ~(...)    → Python's NOT: ~x = -(x+1) → back to signed negative

  Example: a = 0xFFFFFFFF (which is -1 in 32-bit)
  a ^ MASK = 0xFFFFFFFF ^ 0xFFFFFFFF = 0
  ~0 = -1  ✓

  Example: a = 0xFFFFFFF8 (which is -8 in 32-bit)
  a ^ MASK = 0xFFFFFFF8 ^ 0xFFFFFFFF = 7
  ~7 = -8  ✓
```

In [4]:
print(f"Most negative  0x80000000 = {0x80000000}")   # as Python unsigned
print(f"-1             0xFFFFFFFF = {0xFFFFFFFF}")
print(f" 0             0x00000000 = {0x00000000}")
print(f"+1             0x00000001 = {0x00000001}")
print(f"Most positive  0x7FFFFFFF = {0x7FFFFFFF}")

Most negative  0x80000000 = 2147483648
-1             0xFFFFFFFF = 4294967295
 0             0x00000000 = 0
+1             0x00000001 = 1
Most positive  0x7FFFFFFF = 2147483647


In [ ]:
MASK = 0xFFFFFFFF
MAX  = 0x7FFFFFFF

# Show what the mask does
big_number = 0x1FFFFFFFF   # 33 bits — one bit too many
print(f"Before mask: {big_number:b}  ({big_number})")
print(f"After  mask: {big_number & MASK:032b}  ({big_number & MASK})")
print()

# Show the convert-back formula
for a_32bit, expected in [(0xFFFFFFFF, -1), (0xFFFFFFF8, -8), (0xFFFFFFFD, -3)]:
    result = ~(a_32bit ^ MASK)
    status = "✅" if result == expected else "❌"
    print(f"{status}  0x{a_32bit:X}  →  ~(a ^ MASK) = {result}  (expected {expected})")

<a id='5'></a>
## 5. 🧩 Pattern 1: Single Number — LC 136

---

```
  PROBLEM:
  Every element appears TWICE except one. Find the lone element.
  O(n) time, O(1) space — no hash map allowed.

  TRICK: XOR everything. Pairs cancel. Lone survivor remains.

  WHY IT WORKS:
  You have a pool of numbers. Every duplicate is a pair.
  Pairs → zero. Zero XOR lone = lone.

  [4, 1, 2, 1, 2]
  0 ^ 4 ^ 1 ^ 2 ^ 1 ^ 2
  = 0 ^ (1^1) ^ (2^2) ^ 4    ← associativity lets you reorder
  = 0 ^   0   ^   0   ^ 4
  = 4  ✓
```

In [5]:
from typing import List

def singleNumber(nums: List[int]) -> int:
    """
    LC 136 — Single Number
    XOR all values together. Pairs cancel to 0.
    The lone element has no partner — it survives.
    Time: O(n)  Space: O(1)
    """
    result = 0
    for n in nums:
        result ^= n    # pairs will cancel each other out
    return result      # only the lone element remains


def test_harness(fn):
    tests = [
        ([2, 2, 1],          1),
        ([4, 1, 2, 1, 2],    4),
        ([1],                1),
        ([0, 1, 0],          1),
        ([99],               99),
    ]
    passed = 0
    for nums, expected in tests:
        got = fn(nums)
        status = "✅" if got == expected else "❌"
        print(f"{status}  nums={nums}  expected={expected}  got={got}")
        passed += (got == expected)
    print(f"\nResult: {passed}/{len(tests)} passed")

test_harness(singleNumber)

✅  nums=[2, 2, 1]  expected=1  got=1
✅  nums=[4, 1, 2, 1, 2]  expected=4  got=4
✅  nums=[1]  expected=1  got=1
✅  nums=[0, 1, 0]  expected=1  got=1
✅  nums=[99]  expected=99  got=99

Result: 5/5 passed


<a id='6'></a>
## 6. 🧩 Pattern 2: Missing Number — LC 268

---

```
  PROBLEM:
  Array of n distinct numbers from range [0..n]. One is missing. Find it.

  TWO APPROACHES:

  ── APPROACH A: Gauss Formula (simplest) ──
  Expected sum of [0..n] = n*(n+1)//2
  Actual sum = sum(nums)
  Missing = expected - actual

  ── APPROACH B: XOR ──
  Construct a pool where every number appears TWICE — once as an index,
  once as a value. The missing number only appears as an index (no value
  partner). It survives.

  Seed with n (covers the index n which is never in the array).
  XOR in every index i AND every value nums[i].

  Example: [3, 0, 1], missing = 2, n = 3

  Pool built:  n=3,  i=0,v=3,  i=1,v=0,  i=2,v=1
  = 3 ^ 0^3 ^ 1^0 ^ 2^1
  = (3^3) ^ (0^0) ^ (1^1) ^ 2
  =   0   ^   0   ^   0   ^ 2
  = 2  ✓

  WHY DOES n NOT HAVE A PAIR?
  Indices go 0..n-1 PLUS we seed n. Values go 0..n-1 (missing one).
  The missing value is the only one without a matching index partner.
```

In [7]:
def missingNumber(nums: List[int]) -> int:
    """
    LC 268 — Missing Number
    Approach A: Gauss. Expected sum minus actual sum = missing.
    Time: O(n)  Space: O(1)
    """
    n = len(nums)
    return n * (n + 1) // 2 - sum(nums)   # expected sum - actual sum


def missingNumber_xor(nums: List[int]) -> int:
    """
    LC 268 — Missing Number
    Approach B: XOR. Seed with n, XOR every index and value.
    Pairs cancel. Missing value survives.
    Time: O(n)  Space: O(1)
    """
    res = len(nums)                    # seed — index n has no value partner
    for i, num in enumerate(nums):
        res ^= i ^ num                 # index and value pair up and cancel
    return res                         # missing value never got a partner


def test_missing(fn):
    tests = [
        ([3, 0, 1],          2),
        ([0, 1],             2),
        ([9,6,4,2,3,5,7,0,1], 8),
        ([8,6,4,2,3,5,7,0,1], 9),
        ([0],                1),
        ([1],                0),
    ]
    passed = 0
    for nums, expected in tests:
        got = fn(nums)
        status = "✅" if got == expected else "❌"
        print(f"{status}  nums={nums}  expected={expected}  got={got}")
        passed += (got == expected)
    print(f"\nResult: {passed}/{len(tests)} passed")

print("=== Gauss ===")
test_missing(missingNumber)
print("\n=== XOR ===")
test_missing(missingNumber_xor)

=== Gauss ===
✅  nums=[3, 0, 1]  expected=2  got=2
✅  nums=[0, 1]  expected=2  got=2
✅  nums=[9, 6, 4, 2, 3, 5, 7, 0, 1]  expected=8  got=8
✅  nums=[8, 6, 4, 2, 3, 5, 7, 0, 1]  expected=9  got=9
✅  nums=[0]  expected=1  got=1
✅  nums=[1]  expected=0  got=0

Result: 6/6 passed

=== XOR ===
✅  nums=[3, 0, 1]  expected=2  got=2
✅  nums=[0, 1]  expected=2  got=2
✅  nums=[9, 6, 4, 2, 3, 5, 7, 0, 1]  expected=8  got=8
✅  nums=[8, 6, 4, 2, 3, 5, 7, 0, 1]  expected=9  got=9
✅  nums=[0]  expected=1  got=1
✅  nums=[1]  expected=0  got=0

Result: 6/6 passed


<a id='7'></a>
## 7. 🧩 Pattern 3: Number of 1 Bits — LC 191

---

```
  PROBLEM: Count how many 1-bits are in an integer (Hamming weight).

  APPROACH A: Shift and mask
  Read the last bit with n & 1. Then shift n right to expose the next bit.
  Repeat 32 times.

  APPROACH B: Brian Kernighan's trick
  n & (n-1) clears the LOWEST set bit.
  Count how many times you can do this before n hits zero.
  Only iterates as many times as there are 1-bits — faster for sparse numbers.

  WHY n & (n-1) clears the lowest set bit:
  n     = ...1 0 0 0   (lowest set bit at position k, zeros below)
  n-1   = ...0 1 1 1   (borrows down through zeros, flips them)
  n&(n-1)= ..0 0 0 0   (bit k and below all cleared)

  Example: n = 12 (1100)
  12 & 11 = 1100 & 1011 = 1000  (cleared bit 2)
   8 &  7 = 1000 & 0111 = 0000  (cleared bit 3)
  → 2 iterations → 2 set bits  ✓
```

In [10]:
def hammingWeight(n: int) -> int:
    """
    LC 191 — Number of 1 Bits
    Brian Kernighan: n & (n-1) clears lowest set bit each round.
    Count rounds until n is zero.
    Time: O(k) where k = number of set bits   Space: O(1)
    """
    count = 0
    while n:                    # keep going while any bit is still set
        n &= n - 1              # wipe out the lowest 1-bit
        count += 1              # that was one more set bit
    return count


def test_hamming(fn):
    tests = [
        (0b00000000000000000000000000001011, 3),
        (0b00000000000000000000000010000000, 1),
        (0b11111111111111111111111111111101, 31),
        (0,  0),
        (1,  1),
        (15, 4),
    ]
    passed = 0
    for n, expected in tests:
        got = fn(n)
        status = "✅" if got == expected else "❌"
        print(f"{status}  n={n:032b}  expected={expected}  got={got}")
        passed += (got == expected)
    print(f"\nResult: {passed}/{len(tests)} passed")

test_hamming(hammingWeight)

✅  n=00000000000000000000000000001011  expected=3  got=3
✅  n=00000000000000000000000010000000  expected=1  got=1
✅  n=11111111111111111111111111111101  expected=31  got=31
✅  n=00000000000000000000000000000000  expected=0  got=0
✅  n=00000000000000000000000000000001  expected=1  got=1
✅  n=00000000000000000000000000001111  expected=4  got=4

Result: 6/6 passed


In [9]:
def hammingWeight(n: int) -> int:
    """
    LC 191 — Number of 1 Bits (shift + mask version)
    Read n one bit at a time from right to left.
    n & 1 grabs the last bit. n >>= 1 slides the window right.
    Do this exactly 32 times — one per bit position.
    Time: O(32) = O(1)   Space: O(1)
    """
    count = 0
    for _ in range(32):
        count += n & 1      # last bit is 1? add it to count
        n >>= 1             # slide right — expose the next bit
    return count

# Slow motion on n = 13 (1101):
# round  n (binary)   n&1   count
#   1    ...1101        1       1
#   2    ...0110        0       1
#   3    ...0011        1       2
#   4    ...0001        1       3
#   5+   ...0000        0       3
# final count = 3  ✓

print(hammingWeight(13))   # 3
print(hammingWeight(0))    # 0
print(hammingWeight(15))   # 4

3
0
4


<a id='8'></a>
## 8. 🧩 Pattern 4: Counting Bits — LC 338

---

```
  PROBLEM: Return array where ans[i] = number of 1-bits in i, for 0..n.

  APPROACH: DP with offset (powers of 2)

  Key insight: every number i is a power-of-2 (offset) PLUS some smaller
  number. That smaller number's bit count is already computed in dp.
  Just add 1 for the new leading bit.

  dp[i] = 1 + dp[i - offset]
  offset = last power of 2 seen (update when i == offset * 2)

  Step through [0..8]:
  i=0  offset=1  dp[0]=0           (base)
  i=1  offset=1  dp[1]=1+dp[0]=1   (1 = 1 + 0)
  i=2  offset=2  dp[2]=1+dp[0]=1   (2 = 2 + 0)
  i=3  offset=2  dp[3]=1+dp[1]=2   (3 = 2 + 1)
  i=4  offset=4  dp[4]=1+dp[0]=1   (4 = 4 + 0)
  i=5  offset=4  dp[5]=1+dp[1]=2   (5 = 4 + 1)
  i=6  offset=4  dp[6]=1+dp[2]=2   (6 = 4 + 2)
  i=7  offset=4  dp[7]=1+dp[3]=3   (7 = 4 + 3)
  i=8  offset=8  dp[8]=1+dp[0]=1   (8 = 8 + 0)

  Result: [0,1,1,2,1,2,2,3,1]  ✓
```

In [11]:
def countBits(n: int) -> List[int]:
    """
    LC 338 — Counting Bits
    DP with offset. Each number = nearest power-of-2 + remainder.
    dp[i] = 1 + dp[i - offset]. Update offset when i hits next power of 2.
    Time: O(n)   Space: O(1) extra (output excluded)
    """
    dp = [0] * (n + 1)
    offset = 1                          # tracks the last power of 2 we crossed
    for i in range(1, n + 1):
        if offset * 2 == i:             # i is a new power of 2 — update fence
            offset = i
        dp[i] = 1 + dp[i - offset]     # leading bit (1) + tail bit count
    return dp


def test_count_bits(fn):
    tests = [
        (0, [0]),
        (1, [0, 1]),
        (2, [0, 1, 1]),
        (5, [0, 1, 1, 2, 1, 2]),
        (8, [0, 1, 1, 2, 1, 2, 2, 3, 1]),
    ]
    passed = 0
    for n, expected in tests:
        got = fn(n)
        status = "✅" if got == expected else "❌"
        print(f"{status}  n={n}  expected={expected}  got={got}")
        passed += (got == expected)
    print(f"\nResult: {passed}/{len(tests)} passed")

test_count_bits(countBits)

✅  n=0  expected=[0]  got=[0]
✅  n=1  expected=[0, 1]  got=[0, 1]
✅  n=2  expected=[0, 1, 1]  got=[0, 1, 1]
✅  n=5  expected=[0, 1, 1, 2, 1, 2]  got=[0, 1, 1, 2, 1, 2]
✅  n=8  expected=[0, 1, 1, 2, 1, 2, 2, 3, 1]  got=[0, 1, 1, 2, 1, 2, 2, 3, 1]

Result: 5/5 passed


<a id='9'></a>
## 9. 🧩 Pattern 5: Reverse Bits — LC 190

---

```
  PROBLEM: Reverse the 32 bits of an unsigned integer.

  APPROACH: Shift and build
  Loop 32 times. Each iteration:
  1. Grab the last bit of n with n & 1
  2. Shift result left to make room: result <<= 1
  3. OR the grabbed bit into result: result |= (n & 1)
  4. Drop the last bit of n: n >>= 1

  You're reading n from LSB to MSB and writing into result MSB to LSB.
  That's the reversal.

  Example: n = 0b1011 (just 4 bits for clarity)

  Step 1: grab 1, result = 0001
  Step 2: grab 1, result = 0011
  Step 3: grab 0, result = 0110
  Step 4: grab 1, result = 1101

  Input:  1011
  Output: 1101  ✓ (reversed)
```

In [12]:
def reverseBits(n: int) -> int:
    """
    LC 190 — Reverse Bits
    Read n right-to-left (LSB first), write into result left-to-right.
    32 iterations — one per bit position.
    Time: O(1) — always 32 iterations   Space: O(1)
    """
    result = 0
    for _ in range(32):
        result <<= 1            # shift result left — make room for next bit
        result |= (n & 1)       # grab last bit of n, drop it into result
        n >>= 1                 # drop that bit from n, expose the next one
    return result


def test_reverse_bits(fn):
    tests = [
        (0b00000010100101000001111010011100, 964176192),
        (0b11111111111111111111111111111101, 3221225471),
        (0,  0),
        (1,  2147483648),
    ]
    passed = 0
    for n, expected in tests:
        got = fn(n)
        status = "✅" if got == expected else "❌"
        print(f"{status}  n={n:032b}  expected={expected}  got={got}")
        passed += (got == expected)
    print(f"\nResult: {passed}/{len(tests)} passed")

test_reverse_bits(reverseBits)

✅  n=00000010100101000001111010011100  expected=964176192  got=964176192
✅  n=11111111111111111111111111111101  expected=3221225471  got=3221225471
✅  n=00000000000000000000000000000000  expected=0  got=0
✅  n=00000000000000000000000000000001  expected=2147483648  got=2147483648

Result: 4/4 passed


<a id='10'></a>
## 10. 🧩 Pattern 6: Sum Without + or - — LC 371

---

```
  PROBLEM: Return a + b using only bit operations.

  THE INSIGHT: Binary addition is two operations:
  1. XOR  = add the bits, ignore carry
  2. AND << 1 = carry (where both bits were 1, carry shifts left one position)

  Keep feeding carry back in until there's nothing left to carry.

  Example: 3 + 5 = 0011 + 0101
  Round 1: XOR = 0110 (6), carry = (0001) << 1 = 0010 (2)
  Round 2: XOR = 0100 (4), carry = (0010) << 1 = 0100 (4)
  Round 3: XOR = 0000 (0), carry = (0100) << 1 = 1000 (8)
  Round 4: XOR = 1000 (8), carry = 0
  → result = 8  ✓

  PYTHON PROBLEM: No 32-bit ceiling → infinite carry chains for negatives.
  FIX: mask every intermediate result to 32 bits.
  CONVERT BACK: ~(a ^ MASK) restores Python signed int from 32-bit encoding.

  MASK = 0xFFFFFFFF  — chops at bit 31
  MAX  = 0x7FFFFFFF  — if result > MAX, bit 31 is set → negative
```

In [14]:
def getSum(a: int, b: int) -> int:
    """
    LC 371 — Sum of Two Integers
    XOR = sum without carry. AND<<1 = carry. Loop until carry is zero.
    MASK enforces 32-bit ceiling (Python ints are infinite — this fixes that).
    Time: O(1) — at most 32 iterations   Space: O(1)
    """
    MASK = 0xFFFFFFFF   # 32-bit fence — chops off any bits above position 31
    MAX  = 0x7FFFFFFF   # highest positive 32-bit int — sign bit is 0

    while b & MASK:                       # keep going while carry bits exist
        carry = ((a & b) << 1) & MASK     # AND finds carry spots, shift moves them left
        a = (a ^ b) & MASK                # XOR adds without carry, mask keeps 32-bit
        b = carry                         # carry becomes the new b — absorb next round

    # if a > MAX, bit 31 is set — negative in 32-bit world
    # ~(a ^ MASK) flips back to Python's signed negative representation
    return a if a <= MAX else ~(a ^ MASK)


def test_get_sum(fn):
    tests = [
        (1,  2,    3),
        (3,  5,    8),
        (-1, 1,    0),
        (-3, -5,  -8),
        (1000, 1000, 2000),
        (0,  0,    0),
    ]
    passed = 0
    for a, b, expected in tests:
        got = fn(a, b)
        status = "✅" if got == expected else "❌"
        print(f"{status}  {a} + {b} = {got}  (expected {expected})")
        passed += (got == expected)
    print(f"\nResult: {passed}/{len(tests)} passed")

test_get_sum(getSum)

✅  1 + 2 = 3  (expected 3)
✅  3 + 5 = 8  (expected 8)
✅  -1 + 1 = 0  (expected 0)
✅  -3 + -5 = -8  (expected -8)
✅  1000 + 1000 = 2000  (expected 2000)
✅  0 + 0 = 0  (expected 0)

Result: 6/6 passed


<a id='11'></a>
## 11. 🗺️ The Bit Trick Decision Map

---

```
  SIGNAL IN THE PROBLEM                   WHAT TO USE
  ──────────────────────────────────────────────────────────────
  "one appears once, rest appear twice"   XOR everything → lone survivor
  "find missing number in [0..n]"         Gauss (n*(n+1)//2 - sum) OR XOR
  "count 1-bits / Hamming weight"         n & (n-1) loop OR bin(n).count('1')
  "bit count for all i in [0..n]"         DP with offset: dp[i]=1+dp[i-offset]
  "reverse the bits"                      Shift-and-build loop (32 iterations)
  "add without + or -"                    XOR + carry simulation + 32-bit mask
  "is power of 2?"                        n > 0 and (n & (n-1)) == 0
  "isolate lowest set bit"                n & (-n)  OR  n & (n-1) to clear it
  ──────────────────────────────────────────────────────────────

  THE CORE IDENTITIES — MEMORIZE THESE:

  x ^ x = 0           same number cancels
  x ^ 0 = x           XOR with zero is identity
  x & 1               read last bit (0 or 1)
  x >> 1              drop last bit (floor divide by 2)
  x << 1              double x (multiply by 2)
  n & (n-1)           clear lowest set bit
  n & (-n)            isolate lowest set bit
  ~x = -(x+1)         Python NOT
  x & MASK            enforce 32-bit ceiling
  ~(x ^ MASK)         convert 32-bit unsigned back to Python signed

  WHEN YOU SEE NEGATIVES + BIT OPS:
  Always add the 32-bit mask. Python's infinite precision will bite you.
  MASK = 0xFFFFFFFF
  MAX  = 0x7FFFFFFF
```

<a id='12'></a>
## 12. 📋 Interview Cheat Sheet

---

### Quick reference — operators:

| Operator | Symbol | Use it for |
|---|---|---|
| AND | `&` | Mask bits, check if bit is set, isolate |
| OR | `\|` | Set a bit, combine two bit patterns |
| XOR | `^` | Toggle, cancel pairs, find difference |
| NOT | `~` | Flip all bits. `~x = -(x+1)` in Python |
| Left shift | `<<` | Multiply by 2^n, move carry left |
| Right shift | `>>` | Floor divide by 2^n, drop last bit |

---

### The patterns by LC number:

| LC | Problem | Core Trick |
|---|---|---|
| 136 | Single Number | `result ^= n` — pairs cancel |
| 268 | Missing Number | Gauss or XOR with index pairing |
| 191 | Number of 1 Bits | `n &= n-1` — clears lowest set bit |
| 338 | Counting Bits | `dp[i] = 1 + dp[i - offset]` |
| 190 | Reverse Bits | Shift-and-build 32 rounds |
| 371 | Sum Without +/- | XOR + carry + 32-bit mask |

---

### Gotchas — do not forget:

```
❌  Python ints are infinite precision — negatives loop forever without MASK
❌  ~x is NOT the same as flipping 32 bits — ~x = -(x+1) always in Python
❌  n & (n-1) on n=0 is fine (returns 0) but your while loop must check n first
✅  XOR is commutative + associative — order of XOR never matters
✅  n & (n-1) == 0 is the fastest "is power of 2" check (also check n > 0)
✅  To convert 32-bit unsigned back to signed: ~(a ^ MASK)
✅  bin(n).count('1') is valid in an interview if you're not told to avoid builtins
```

---

### The 32-bit boilerplate — copy this when negatives are in the problem:

```python
MASK = 0xFFFFFFFF   # 32-bit fence
MAX  = 0x7FFFFFFF   # max positive 32-bit int

# use (x & MASK) on every intermediate result
# at the end:
return a if a <= MAX else ~(a ^ MASK)
```

---
*End of Bit Operations Master Guide — Sean Edition*